# Colab notebook for fine-tuning

In [34]:
%pip install transformers datasets pandas scikit-learn sentencepiece torch accelerate -q

In [35]:
import pandas as pd
import logging
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
LOGGER = logging.getLogger(__name__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
LOGGER.info(f"Using device: {device}")

MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
TASK_PREFIX = "translate slang to standard: "

In [36]:
from transformers import DataCollatorForSeq2Seq

In [41]:
def load_data(train_path, test_path):
    """Load train/test CSVs into Hugging Face Datasets."""
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Drop rows with missing source or target text
    train_df = train_df.dropna(subset=["source_text", "target_text"])
    test_df = test_df.dropna(subset=["source_text", "target_text"])

    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

    LOGGER.info(f"Loaded {len(train_dataset)} training examples")
    LOGGER.info(f"Loaded {len(test_dataset)} test examples")

    return train_dataset, test_dataset


def preprocess_function(examples, tokenizer):
    """Tokenize inputs with task prefix and targets as labels.

    Updated to use modern text_target argument to avoid AttributeError.
    """
    inputs = [TASK_PREFIX + str(ex) for ex in examples["source_text"]]
    targets = [str(t) for t in examples["target_text"]]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    # Tokenize targets using the modern text_target argument
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [38]:
train_dataset, test_dataset = load_data(
    "/content/data/processed/train.csv",
    "/content/data/processed/test.csv"
)

In [39]:
LOGGER.info(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
LOGGER.info(f"Model loaded. Parameters: {model.num_parameters():,.0f}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [42]:
LOGGER.info("Tokenizing training dataset...")
train_dataset = train_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names,
)

LOGGER.info("Tokenizing test dataset...")
test_dataset = test_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=test_dataset.column_names,
)

LOGGER.info(f"Train dataset columns: {train_dataset.column_names}")

# Sanity check: verify labels are NOT all -100
sample_labels = train_dataset[0]["labels"]
valid_tokens = [t for t in sample_labels if t != -100]
LOGGER.info(f"Sample label has {len(valid_tokens)} valid tokens (should be > 0): {valid_tokens[:10]}")
LOGGER.info("Tokenization complete!")

Map:   0%|          | 0/3543 [00:00<?, ? examples/s]

Map:   0%|          | 0/626 [00:00<?, ? examples/s]

Let's inspect a few samples from the tokenized `train_dataset` to understand the `input_ids` and `labels` content. This will help determine if the data is being correctly prepared for training.

In [43]:
LOGGER.info("Inspecting tokenized training data samples...")

# Display the first 3 samples from the tokenized train_dataset
for i in range(3):
    sample = train_dataset[i]
    print(f"--- Sample {i+1} ---")
    print(f"Input IDs: {sample['input_ids']}")
    print(f"Attention Mask: {sample['attention_mask']}")
    print(f"Labels (with -100 for padding): {sample['labels']}")

    # Decode input_ids and labels for better understanding
    decoded_input = tokenizer.decode([id for id in sample['input_ids'] if id != tokenizer.pad_token_id], skip_special_tokens=True)
    # Filter out -100 before decoding labels
    decoded_label = tokenizer.decode([id for id in sample['labels'] if id != -100 and id != tokenizer.pad_token_id], skip_special_tokens=True)
    print(f"Decoded Input: {decoded_input}")
    print(f"Decoded Label: {decoded_label}")
    print("\n")

--- Sample 1 ---
Input IDs: [13959, 3, 7, 4612, 12, 1068, 10, 9161, 48, 5812, 19, 1517, 423, 1472, 827, 1]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Labels (with -100 for padding): [48, 5812, 19, 1517, 326, 1287, 6, 1237, 827, 12855, 1]
Decoded Input: translate slang to standard: bro this professor is giving full fire energy
Decoded Label: this professor is giving off excellent, amazing energy honestly


--- Sample 2 ---
Input IDs: [13959, 3, 7, 4612, 12, 1068, 10, 3, 1825, 78, 25, 54, 31, 17, 36, 20, 40, 83, 76, 81, 48, 3, 52, 29, 1]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Labels (with -100 for padding): [200, 1565, 25, 1178, 36, 78, 271, 20, 7650, 6318, 6, 31820, 4454, 81, 48, 269, 230, 1]
Decoded Input: translate slang to standard: ok so you can't be delulu about this rn
Decoded Label: best friend you cannot be so being delusional, unrealistic expectations about this right now


--- Sample 3 ---
Input IDs: [1

In [44]:
training_args = Seq2SeqTrainingArguments(
    output_dir="backend/fine_tuned_model",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    save_total_limit=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    learning_rate=5e-4,   # Bumped slightly — 1e-4 is on the low end for flan-t5-small fine-tuning
    seed=42,
    fp16=device == "cuda",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

LOGGER.info("Training configuration ready")

In [46]:
# DataCollatorForSeq2Seq pads inputs and labels to the longest sequence in each
# batch, and replaces label padding tokens with -100 so they're ignored in loss.
# This is the ONLY place padding + masking should happen.
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if device == "cuda" else None,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
)

LOGGER.info("Starting training...")
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,7.799902,6.928464
2,5.622615,4.301146
3,2.988015,2.370475
4,2.027165,1.588302
5,1.463157,1.199174
6,1.152299,0.954296
7,0.937821,0.820854
8,0.799344,0.728861
9,0.753615,0.680364
10,0.681724,0.665634


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


TrainOutput(global_step=2220, training_loss=2.646453096200754, metrics={'train_runtime': 9355.4296, 'train_samples_per_second': 3.787, 'train_steps_per_second': 0.237, 'total_flos': 388145182900224.0, 'train_loss': 2.646453096200754, 'epoch': 10.0})

In [47]:
output_dir = "backend/fine_tuned_model"
import shutil

# Clear existing output directory to avoid file locking issues
if Path(output_dir).exists():
    shutil.rmtree(output_dir)

LOGGER.info(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=False)
LOGGER.info("Model and tokenizer saved!")
LOGGER.info("Training complete!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [48]:
# Zip and download the fine-tuned model checkpoint
from google.colab import files
import shutil

model_checkpoint = "backend/fine_tuned_model"
shutil.make_archive("checkpoint", "zip", ".", model_checkpoint)
files.download("checkpoint.zip")
LOGGER.info("Checkpoint zipped and ready for download to Google Drive")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [57]:
# Load the trained model for inference
trained_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
trained_tokenizer = AutoTokenizer.from_pretrained(output_dir)
trained_model.eval()

# Test example
test_input = "wanna chill, im done grinding."
inputs = trained_tokenizer(test_input, return_tensors="pt")
with torch.no_grad():
    outputs = trained_model.generate(**inputs, max_length=128)
result = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Input:  {test_input}")
print(f"Output: {result}")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Input:  wanna chill, im done grinding.
Output: i am going to relax, i am done working.
